# Evaluation (precision, recall, f1-score - Micro, Macro, Weighted) 

## Get entities LLM

In [1]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from get_entities_LLM import extract_llm_entities, entity_vocab
from tqdm import tqdm

#%%
# Load your labeled dataset
df = pd.read_csv("ARP_PreLabel.csv")
df["entities"] = df["Entities"].apply(eval)

#%%
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)
test_sentences = test_df["Sentence"].tolist()
true_entities_list = test_df["entities"].tolist()

#%%
# Wrap original extract_llm_entities with a progress bar (no need to modify .py)
def extract_llm_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="LLM Prediction"):
        result = extract_llm_entities([sent])[0]
        results.append(result)
    return results

In [2]:
#%%
# Run prediction with progress bar
pred_results = extract_llm_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 445/445 [31:14<00:00,  4.21s/it]


In [3]:
#%%
# get entities from tuple
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list]

#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]


In [4]:
#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.6141131595677051
Macro F1: 0.5083618760327107
Weighted F1: 0.6267650001151047

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.65      0.68      0.66        77
           Interest Rates       0.78      0.63      0.70        49
                Inflation       0.95      0.92      0.93        85
               Employment       0.74      0.88      0.81        33
             Unemployment       0.62      1.00      0.76         8
                      GDP       0.48      0.46      0.47        26
                    Trade       0.75      1.00      0.86         6
                 Congress       0.00      0.00      0.00         0
          Monetary Policy       0.54      0.74      0.62        70
      Financial Stability       0.00      0.00      0.00         0
          Price Stability       0.39      0.65      0.49        17
Regulatory Implementation       0.00      0.00      0.00         2
              

/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
